# 17 · The Copilot harness – J.A.R.V.I.S. stops writing his own loop 🧪

> 🧪 **Preview.** The Microsoft Copilot harness is a public preview (announced in [the Foundry blog](https://devblogs.microsoft.com/foundry/microsoft-copilot-harness-in-foundry/)). Names, fields and limits may change. This notebook uses `AIProjectClient(..., allow_preview=True)` and `azure-ai-projects>=2.9.0`.

Until now Tony built every loop himself: the function-call loop in chapter 05, a whole container in chapter 09. The **Copilot harness** is a managed runtime for **prompt agents**. Foundry runs the agent loop and a sandbox with a workspace. You still only write configuration: model, instructions, tools and **skills**. In chapter 07 a skill reference on a default prompt agent did nothing; on the harness it works. And a skill is more than text: `invoice-review` ships five **scripts**, and the harness runs them in its sandbox.

| | Default prompt agent | Prompt agent + Copilot harness 🧪 | Hosted agent |
|---|---|---|---|
| You write | definition | definition (`harness=MicrosoftCopilotHarness()`) | code |
| Agent loop | Foundry | Foundry (the harness) | your code |
| Skills (`skills=[SkillReference]`) | ❌ accepted, no effect | ✅ | your code loads them |
| Skill scripts | ❌ | ✅ run with the shell tool | your code runs them |
| Built-in tools | none | file read, file write, shell, web fetch, subagents | whatever you code |
| Workspace (files between turns) | ❌ | ✅ one per conversation | sessions |
| Deploy | `create_version` | `create_version` or `azd deploy` | `azd deploy` |

> ⚠️ **Before you start.** The project region must support the harness (Sweden Central, this repo's default, does), and the model must support reasoning and tool calling. You choose the harness when you **create** an agent and cannot switch it later. The harness runtime is free until 2026-12-01; after that the sandbox is billed per vCPU-hour and GiB-hour, on top of model tokens.

Deep dive: [docs/copilot-harness.md](../docs/copilot-harness.md).

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *
from stark_docs import extract_invoice      # Document Intelligence prebuilt-invoice (chapter 13)

import io, re, json, zipfile
from importlib.metadata import version
from azure.ai.projects.models import (
    CreateSkillVersionFromFilesBody, MicrosoftCopilotHarness, MicrosoftCopilotToolsetConfig,
    MicrosoftCopilotToolsetPreview, PromptAgentDefinition, SkillReference,
)

print("azure-ai-projects", version("azure-ai-projects"))   # needs 2.9.0 or later for MicrosoftCopilotHarness
pp = project(preview=True)                                    # allow_preview=True: skills and the harness are preview
SDK_AGENT = "jarvis-nb17"        # created here with the SDK
AZD_AGENT = "jarvis-harness"     # deployed with azd in section 6 (name from src/jarvis-harness/azure.yaml)

## 1 · Publish the skills as packages
The harness reads skills from Foundry, not from disk. A skill is a **folder**: `SKILL.md` plus anything it needs. Chapter 07 uploaded only the text (`inline_content`). Here we zip each folder and upload it with `create_from_files`, so the scripts and data files travel with the instructions:

```
skills/invoice-review/
├── SKILL.md                 checklist + the exact commands to run
├── scripts/                 check_invoice.py, vendor_check.py, iban_check.py, ledger.py, approval_limit.py
└── assets/                  vendors.csv, approval-limits.csv, invoice-example.json
```

Each upload is a new immutable version. We keep the version numbers to pin them on the agent.

In [ ]:
def publish_skill(name: str):
    """Zip skills/<name>/ with SKILL.md at the root of the archive and upload it as a new skill version."""
    folder = ROOT / "skills" / name
    files = sorted(p for p in folder.rglob("*") if p.is_file() and "__pycache__" not in p.parts)
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        for p in files:
            z.write(p, p.relative_to(folder).as_posix())
    print(f"{name}: {[p.relative_to(folder).as_posix() for p in files]}")
    return pp.beta.skills.create_from_files(name, content=CreateSkillVersionFromFilesBody(
        files=[(f"{name}.zip", buf.getvalue(), "application/zip")]))

published = {s.name: s.version for s in (publish_skill("stark-escalation"), publish_skill("invoice-review"))}
print(published)

## 2 · Create an agent on the harness
Three things differ from chapter 04:

| Field | Value | Why |
|---|---|---|
| `harness` | `MicrosoftCopilotHarness()` | selects the managed runtime. Fixed for the life of the agent |
| `skills` | `SkillReference(name, version)` | attaches the skills. Each agent version pins one skill version |
| `tools` | one `MicrosoftCopilotToolsetPreview` | switches built-in harness tools on or off. All five are on by default |

This time **shell stays on**: it is how the agent runs the skill's scripts. Shell is the broadest tool (it can read, write and reach the network on its own), so we switch off what J.A.R.V.I.S. does not need: `subagents`. To also limit where the sandbox may connect, attach a network egress policy (preview, not done here, see [docs/copilot-harness.md](../docs/copilot-harness.md)). The harness manages its own settings, so leave out `temperature`, `top_p`, `tool_choice` and `text.format`.

In [ ]:
definition = PromptAgentDefinition(
    model=model("chat"),
    instructions=("You are J.A.R.V.I.S., Tony Stark's assistant in the workshop. "
                  "Use your skills when a request matches them, and run a skill's scripts when it ships them. "
                  "Keep answers short."),
    harness=MicrosoftCopilotHarness(),
    skills=[SkillReference(name=n, version=v) for n, v in published.items()],
    tools=[MicrosoftCopilotToolsetPreview(configs=[
        MicrosoftCopilotToolsetConfig(name="shell", enabled=True),       # runs the invoice-review scripts
        MicrosoftCopilotToolsetConfig(name="subagents", enabled=False),
    ])],
)
agent17 = pp.agents.create_version(agent_name=SDK_AGENT, definition=definition)
print(agent17.name, "version", agent17.version)

stored = pp.agents.get(agent_name=SDK_AGENT).versions.latest.definition.as_dict()   # what Foundry persisted, as plain JSON
print(json.dumps({k: stored.get(k) for k in ("kind", "harness", "skills", "tools")}, indent=2))

## 3 · Ask it, and check that the skill is used
Two rules apply to every harness request:

- every request needs a **conversation** (no `previous_response_id`, no `store=False`)
- use an **agent-bound client** (`get_openai_client(agent_name=...)`), like the Microsoft Learn sample

`stark-escalation` leaves a marker, as in chapter 07: the answer starts with `Severity:`. The output items show what the harness did (a `skill` call, file tools, shell, reasoning).

In [ ]:
INCIDENT = "A suit just started smoking in the hangar. What do I do?"

def ask(agent_name: str, conversation_id: str, text: str, show: int = 400):
    oai = pp.get_openai_client(agent_name=agent_name)
    r = oai.responses.create(conversation=conversation_id, input=text)
    print(f"> {text[:120]}{'...' if len(text) > 120 else ''}\n  output items: {[i.type for i in r.output]}")
    print("  " + r.output_text[:show].replace("\n", "\n  ") + "\n")
    return r

oai17 = pp.get_openai_client(agent_name=SDK_AGENT)
conv17 = oai17.conversations.create()
conversations = [(SDK_AGENT, conv17.id)]

reply = ask(SDK_AGENT, conv17.id, INCIDENT).output_text
print("follows stark-escalation:", bool(re.match(r"\W*severity:", reply, re.I)))

## 4 · A skill that runs scripts
A prompt is good at judgement and bad at arithmetic, checksums and lookups. `invoice-review` therefore ships **prewritten commands**. [SKILL.md](../skills/invoice-review/SKILL.md) tells the agent which one to run for which check. The model only fills in the arguments:

| Check | Command | Fails when |
|---|---|---|
| fields, arithmetic, date | `python scripts/check_invoice.py invoice.json` | a required field is missing, the lines or the total don't add up, the date is in the future |
| vendor master | `python scripts/vendor_check.py "<vendor>"` | the vendor is blocked or unknown ([vendors.csv](../skills/invoice-review/assets/vendors.csv)) |
| bank account | `python scripts/iban_check.py "<IBAN>" --vendor "<vendor>"` | the mod-97 checksum is wrong, or the IBAN is not the one on file |
| duplicate | `python scripts/ledger.py invoice.json` | the vendor's invoice number is already in `ledger.csv` (in the workspace) |
| approval limit | `python scripts/approval_limit.py --approver "<name>" --total <total>` | the total is above the approver's limit ([approval-limits.csv](../skills/invoice-review/assets/approval-limits.csv)) |

Every script prints one JSON line and exits with `0` (pass) or `1` (fail). Standard library only, so they need nothing installed. They are plain CLI tools, so try them here first, on your machine:

In [ ]:
S = "skills/invoice-review/scripts"
sh(f"python {S}/check_invoice.py skills/invoice-review/assets/invoice-example.json")
sh(f'python {S}/vendor_check.py "Oscorp Materials"')
sh(f'python {S}/iban_check.py "GB82 WEST 1234 5698 7654 32" --vendor "Pym Particle Supply Co."')
sh(f'python {S}/approval_limit.py --approver "Happy Hogan" --total 5726.34')

Now the same commands, run by the agent in its sandbox. **Who does what:**

- This notebook reads the PDFs with **Document Intelligence** (`extract_invoice` from chapter 13, with your identity) and sends the extracted fields as JSON.
- The agent saves them as `invoice.json` in its workspace and runs the scripts with the shell tool.

> ❓ The sandbox calling Document Intelligence itself, with the **agent identity**, is not documented for scripts: Foundry uses the agent identity for tools (toolbox, OpenAPI, file upload), and Microsoft Learn shows no way for a sandbox script to get a token. So extraction stays outside the sandbox.

Four invoices, each built to trip a different script. They run **in one conversation**, so the ledger from invoice A is still there for invoice C:

| Case | Invoice | Trap | Expected |
|---|---|---|---|
| A | `invoice-pym-4471.pdf`, IBAN on file | none | `APPROVE` |
| B | `invoice-oscorp-9930.pdf` | Oscorp is blocked in the vendor master | `REJECT` |
| C | `invoice-pym-4471.pdf` again, "we changed banks" | duplicate invoice number and a new, valid IBAN | `REJECT` |
| D | Pym 4502 typed in by hand, approver Happy Hogan | the total has a typo (5,726.34 instead of 5,276.34) | `REJECT` |

In [ ]:
pym = extract_invoice("data/docs/invoice-pym-4471.pdf")
oscorp = extract_invoice("data/docs/invoice-oscorp-9930.pdf")
typo = {"fields": {"VendorName": "Pym Particle Supply Co.", "InvoiceId": "4502", "InvoiceDate": "2026-10-20",
                   "SubTotal": 4885.50, "TotalTax": 390.84, "InvoiceTotal": 5726.34},
        "items": [{"description": "Shrink/grow regulator", "quantity": 4, "unitprice": 1200.0, "amount": 4800.0},
                  {"description": "Shipping and handling", "quantity": 1, "unitprice": 85.5, "amount": 85.5}]}
for name, inv in [("pym", pym), ("oscorp", oscorp)]:
    print(name, inv["fields"]["VendorName"], inv["fields"]["InvoiceId"], inv["fields"]["InvoiceTotal"])

ALL = {"check_invoice", "vendor_check", "iban_check", "ledger", "approval_limit"}
CASES = [  # case, invoice, approver, IBAN from the vendor's email, extra text, expected verdict, scripts that must run
    ("A", pym, "Pepper Potts", "DE89 3704 0044 0532 0130 00", "", "APPROVE", ALL),
    ("B", oscorp, "Pepper Potts", None, "", "REJECT", {"vendor_check"}),
    ("C", pym, "Pepper Potts", "GB82 WEST 1234 5698 7654 32",
     "The vendor writes: we changed banks, please pay this invoice to our new account.", "REJECT", {"iban_check", "ledger"}),
    ("D", typo, "Happy Hogan", None, "", "REJECT", {"check_invoice"}),
]

def invoice_request(inv: dict, approver: str, iban: str | None, extra: str) -> str:
    parts = [f"Review this invoice. Approver: {approver}.", extra]
    if iban:
        parts.append(f"Payment IBAN from the vendor's email: {iban}")
    parts.append("Fields extracted by Document Intelligence:\n" + json.dumps(inv, default=str))
    return "\n".join(p for p in parts if p)

def verdict(text: str) -> str | None:
    found = re.findall(r"APPROVE|REJECT|NEEDS HUMAN REVIEW", text)
    return found[-1] if found else None

def scripts_run(response) -> set[str]:
    """The skill scripts named anywhere in the output items (shell calls and their output)."""
    return set(re.findall(r"scripts/(\w+)\.py", response.model_dump_json()))

def review(agent_name: str, conversation_id: str, case: tuple):
    name, inv, approver, iban, extra, expected, must_run = case
    r = ask(agent_name, conversation_id, invoice_request(inv, approver, iban, extra))
    ran = scripts_run(r)
    print(f"  case {name}: verdict {verdict(r.output_text)} (expected {expected}) | scripts run: {sorted(ran)}"
          f" | missing: {sorted(must_run - ran) or 'none'}\n")
    return r

for case in CASES:
    review(SDK_AGENT, conv17.id, case)

What to look for:

- **A** runs all five scripts and approves: arithmetic, vendor, IBAN, ledger and limit all pass.
- **B** stops at `vendor_check.py`: the reason quotes the note from `vendors.csv`.
- **C** is the classic invoice fraud: the IBAN has a valid checksum, so a model "looking at it" finds nothing wrong. `iban_check.py` sees it is not the account on file, and `ledger.py` remembers invoice 4471 from case A.
- **D** fails in `check_invoice.py` with the exact numbers. `approval_limit.py` would also flag it: Happy Hogan has the `default` limit of 5,000.

`missing` lists scripts the agent should have run but didn't. The model decides what to run, so a run can differ; if a case is off, open its trace (section 👀). ❓ Where the harness places the skill's files in the sandbox, and the exact output item type of a shell call, were not checked: `scripts_run` simply searches all output items for `scripts/<name>.py`.

## 5 · The workspace: files that live as long as the conversation
`ledger.py` wrote `ledger.csv` into the **workspace** of this conversation. The workspace sits in a Foundry-managed sandbox and survives between turns. A new conversation starts with an empty one, so the same invoice is no longer a duplicate there.

In [ ]:
ask(SDK_AGENT, conv17.id, "Show me the contents of ledger.csv in your workspace.")

conv17b = oai17.conversations.create()
conversations.append((SDK_AGENT, conv17b.id))
review(SDK_AGENT, conv17b.id, CASES[0])      # invoice A again, in a fresh workspace: no duplicate, APPROVE

The workspace is **session state, not storage**: Foundry may idle or replace the sandbox, and the workspace is gone with the conversation. A real ledger belongs in a database the scripts can reach. To hand a file back to the user, the harness uploads it to Foundry Files with the **agent identity**, which first needs the **Foundry User** role on the project (not done here, see [docs/copilot-harness.md](../docs/copilot-harness.md)).

## 6 · The same agent as a file, deployed with azd
In the code-first world the agent lives next to the skills it uses. [src/jarvis-harness/azure.yaml](../src/jarvis-harness/azure.yaml) declares:

| Service | Host | Does |
|---|---|---|
| `ai-project` | `azure.ai.project` | points at the STARK project (`STARK_PROJECT_ENDPOINT`), creates nothing |
| `stark-escalation`, `invoice-review` | `azure.ai.skill` | publish the folder `skills/<name>` (scripts and assets included) as a new skill version (`archive:`) |
| `jarvis-harness` | `azure.ai.agent`, `kind: prompt` | the agent: `harness.type: microsoft_copilot_preview`, `skills:`, the toolset with shell on |

It is its **own azd project**: the harness needs `azd` 1.37.0+, `azure.ai.agents` 1.0.0-beta.22+ and the opt-in flag `AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS=true`, while the root [azure.yaml](../azure.yaml) pins older extensions for the hosted agents.

In [ ]:
print((ROOT / "src" / "jarvis-harness" / "azure.yaml").read_text())

Run once in a terminal, or let the next cell do it:

```bash
azd extension upgrade microsoft.foundry          # azure.ai.agents >= 1.0.0-beta.22
python scripts/harness_env.py                    # azd env `stark-harness` in src/jarvis-harness, filled from .env
cd src/jarvis-harness
export AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS=true    # PowerShell: $env:AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS = "true"
azd deploy --all --no-prompt                     # predeploy hook copies skills/, then: skills, then the agent
azd ai agent invoke --new-conversation --no-prompt "A suit just started smoking in the hangar. What do I do?"
```

> ⚠️ After the upgrade, the root `azure.yaml` (which pins `azure.ai.agents` `=1.0.0-beta.20`) may refuse to run. To go back for chapters 08, 09 and 15, reinstall that version: `azd extension install azure.ai.agents --version 1.0.0-beta.20 --force`.

In [ ]:
import os
os.environ["AZD_ALPHA_ENABLE_AGENTS_MANAGED_HARNESS"] = "true"   # the azd opt-in; inherited by sh()

sh("azd ai agent version")
sh("python scripts/harness_env.py")
sh("cd src/jarvis-harness && azd deploy --all --no-prompt")

Check what Foundry stored: `kind: prompt`, `harness.type: microsoft_copilot_preview`, two skill references with **pinned** versions, and the toolset.

In [ ]:
deployed = pp.agents.get(agent_name=AZD_AGENT).versions.latest
d = deployed.definition.as_dict()   # model objects -> plain JSON
print("version", deployed.version)
print(json.dumps({k: d.get(k) for k in ("kind", "harness", "skills", "tools")}, indent=2))

## 7 · Call the deployed agent
Same client, same rules: an agent-bound client and a conversation. Invoice A runs all five scripts again, in the deployed agent's own sandbox.

In [ ]:
oai_azd = pp.get_openai_client(agent_name=AZD_AGENT)
conv_azd = oai_azd.conversations.create()
conversations.append((AZD_AGENT, conv_azd.id))
reply = ask(AZD_AGENT, conv_azd.id, INCIDENT).output_text
print("follows stark-escalation:", bool(re.match(r"\W*severity:", reply, re.I)))
review(AZD_AGENT, conv_azd.id, CASES[0])

## 👀 In the portal
- **Build → Agents**: `jarvis-nb17` and `jarvis-harness`. The agent page shows the harness, **Harness tools** (subagents off) and the two skills.
- Open a response and select **Traces**: the tool calls include `skill`, then the shell commands `python .../scripts/*.py` with their JSON output.
- ❓ Where the portal lists skills on their own (**Tools → Skills (Preview)** in the Microsoft Learn docs) and whether it shows a package's files was not checked.

> 💡 All pages: [docs/portal-tour.md](../docs/portal-tour.md)

## 🧹 Clean up
Deleting a conversation also ends its harness session and workspace. Individual `responses.delete` is not supported on the harness. The skills stay, chapter 07 and the hosted agents use them too.

In [ ]:
for agent_name, cid in conversations:
    try: pp.get_openai_client(agent_name=agent_name).conversations.delete(cid)
    except Exception as exc: print("skip conversation", cid, type(exc).__name__)
delete_agent(SDK_AGENT)
delete_agent(AZD_AGENT)      # or, in src/jarvis-harness: azd ai agent delete --force --no-prompt

## 🎯 Recap
The Copilot harness turns a prompt agent into an agent with a managed loop, a workspace, built-in tools and **skills that work**, without a container. A skill can carry **scripts**: the agent runs the prewritten commands with the shell tool, and the deterministic checks (arithmetic, checksums, lookups, duplicates) no longer depend on the model's maths. You choose the harness at creation, give every request a conversation and switch off the built-in tools you do not need. The same agent fits in an `azure.yaml` next to its skills, so `azd deploy --all` ships both.

| Need | Pick |
|---|---|
| instructions + Foundry tools are enough | default prompt agent (chapter 04) |
| skills with scripts, files, multi-step tool work, no code to host | prompt agent + Copilot harness (this chapter) 🧪 |
| your own framework, loop or dependencies | hosted agent (chapter 09) |

> 💡 Next: compare with `jarvis-skills` in [09 · Hosted agent](09_hosted_agent.ipynb), the same skills with a loop you own. When you are done, remove the workshop with `azd down --purge` in the repo root.